# TKD Vision – Augmentation du dataset
Ce notebook applique des transformations sur tes images originales pour enrichir le dataset.

## 1. Imports et configuration

In [ ]:
import os
import random
from pathlib import Path 
from PIL import Image, ImageEnhance
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ─── CONFIGURATION ───────────────────────────────────────────────
DATASET_DIR   = "dataset"
CLASSES       = ["ap_tchagui", "yop_tchagui"]
IMAGES_CIBLES = 150
SEED          = 42
random.seed(SEED)
# ─────────────────────────────────────────────────────────────────

print('✅ Imports OK')

## 2. Vérification du dataset initial

In [ ]:
print('📂 État du dataset AVANT augmentation\n')
for classe in CLASSES:
    dossier = Path(DATASET_DIR) / classe
    images  = list(dossier.glob('*.jpg'))
    print(f'  {classe} : {len(images)} image(s)')

## 3. Définition des transformations

In [ ]:
def augmenter_image(image_pil):
    """Applique des transformations aléatoires sur une image PIL."""

    # Flip horizontal
    if random.random() > 0.5:
        image_pil = image_pil.transpose(Image.FLIP_LEFT_RIGHT)

    # Rotation légère
    angle = random.uniform(-15, 15)
    image_pil = image_pil.rotate(angle, fillcolor=(0, 0, 0))

    # Luminosité
    facteur = random.uniform(0.7, 1.3)
    image_pil = ImageEnhance.Brightness(image_pil).enhance(facteur)

    # Contraste
    facteur = random.uniform(0.7, 1.3)
    image_pil = ImageEnhance.Contrast(image_pil).enhance(facteur)

    # Zoom léger (crop + resize)
    if random.random() > 0.5:
        w, h   = image_pil.size
        margin = int(min(w, h) * 0.1)
        left   = random.randint(0, margin)
        top    = random.randint(0, margin)
        right  = w - random.randint(0, margin)
        bottom = h - random.randint(0, margin)
        image_pil = image_pil.crop((left, top, right, bottom)).resize((w, h))

    return image_pil

print('✅ Pipeline Pillow défini')
print(f'   → Objectif : {IMAGES_CIBLES} images par classe')

## 4. Aperçu des transformations sur une image

In [ ]:
exemple_path    = list((Path(DATASET_DIR) / CLASSES[0]).glob('*.jpg'))[0]
image_originale = Image.open(str(exemple_path)).convert('RGB')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Aperçu des augmentations', fontsize=14, fontweight='bold')

axes[0, 0].imshow(image_originale)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for i, ax in enumerate(axes.flat[1:]):
    augmented = augmenter_image(image_originale)
    ax.imshow(augmented)
    ax.set_title(f'Augmentation {i+1}')
    ax.axis('off')

plt.tight_layout()
plt.savefig('images/apercu_augmentations.png', dpi=100, bbox_inches='tight')
plt.close()
print('✅ Aperçu sauvegardé → apercu_augmentations.png')

## 5. Application de l'augmentation sur tout le dataset

In [ ]:
for classe in CLASSES:
    dossier      = Path(DATASET_DIR) / classe
    images_paths = list(dossier.glob('*.jpg'))
    nb_original  = len(images_paths)
    nb_a_generer = IMAGES_CIBLES - nb_original

    print(f'\n🥋 Classe : {classe}')
    print(f'   Images originales : {nb_original}')
    print(f'   Images à générer  : {nb_a_generer}')

    if nb_a_generer <= 0:
        print('   ✅ Objectif déjà atteint.')
        continue

    compteur = 0
    while compteur < nb_a_generer:
        src_path = random.choice(images_paths)
        try:
            img       = Image.open(str(src_path)).convert('RGB')
            augmented = augmenter_image(img)
            augmented.save(str(dossier / f'aug_{compteur:04d}.jpg'))
            compteur += 1
        except Exception as e:
            print(f'   ⚠️ Erreur : {e}')
            continue

    print(f'   ✅ {compteur} images générées')

print('\n🎉 Augmentation terminée !')

## 6. Vérification finale

In [ ]:
print('📂 État du dataset APRÈS augmentation\n')
total = 0
for classe in CLASSES:
    dossier = Path(DATASET_DIR) / classe
    images  = list(dossier.glob('*.jpg'))
    total  += len(images)
    print(f'  {classe} : {len(images)} images')

print(f'\n  Total dataset : {total} images')
print('\n➡️  Prochaine étape : split train / val / test')